# M3.5 · 深度学习（LSTM / Transformer / GNN）

> 深度网络能拟合非线性、捕捉序列与关联结构。本机未装 torch，本节用 sklearn MLP 跑真实网络，
> 并用 numpy 手写前馈网络理解原理；LSTM/Transformer/GNN 的结构在此讲透。

## 一、三类深度模型结构

| 模型 | 核心结构 | 量化用途 |
|---|---|---|
| LSTM | 输入门/遗忘门/输出门 + 细胞状态，记忆长序列 | 用历史序列预测下一期 |
| Transformer | 自注意力 self-attention，并行建模全局依赖 | 时序编码、跨资产关系 |
| GNN | 在图上做邻居消息传递 | 股票关系图、供应链/持仓关联 |

**关键概念：**
- LSTM 靠"门控"缓解梯度消失，选择性记忆与遗忘；
- 自注意力 $\text{Attention}(Q,K,V)=\text{softmax}(QK^T/\sqrt{d_k})V$；
- GNN 每一层把邻居信息聚合到节点，多层即多跳关系。

> 数据量不足时深度模型容易过拟合，量化里树模型往往是更强基线，深度模型需严格正则与外推验证。


## 二、加载数据与建模长表

In [ ]:
import sys, numpy as np, pandas as pd
sys.path.insert(0, r'C:\Users\gucao\Desktop\Qlib')
import qlib
from qlib.data import D
from l2.config import load_universe
import config
from factor_lab2 import hfactors
from factor_lab2.combine import cross_zscore
from model_lab2 import labels, deep
qlib.init(provider_uri=str(config.QLIB_DAILY_DIR), joblib_backend='threading')
codes = load_universe('hs300')
raw = D.features(codes, ['$open','$close','$high','$low','$volume'],
                 start_time='2018-01-01', freq='day')
op=raw['$open'].unstack('instrument'); cl=raw['$close'].unstack('instrument')
hi=raw['$high'].unstack('instrument'); lo=raw['$low'].unstack('instrument')
vo=raw['$volume'].unstack('instrument')
lab = labels.cs_rank_label(cl, 5)
hf = hfactors.build_hf_factors(op, cl, hi, lo, vo)
feat_names = ['已实现波动20','日内动量20','尾盘位置20','量价相关20','收益偏度60']
z = pd.concat({n: cross_zscore(hf[n]).stack() for n in feat_names}, axis=1)
z.index.names = ['datetime','instrument']
df = z.join(lab.stack().rename('y')).dropna().sample(60000, random_state=42).sort_index()
X = df[feat_names].values; y = df['y'].values
cut = int(len(df)*0.8)
print('样本:', df.shape)

## 三、sklearn MLP（真实神经网络）

In [ ]:
mlp = deep.make_mlp((32,16))
mlp.fit(X[:cut], y[:cut])
p_mlp = mlp.predict(X[cut:])
ic_mlp = pd.Series(p_mlp).corr(pd.Series(y[cut:]), method='spearman')
print('MLP 测试 IC=%.4f 迭代次数=%d' % (ic_mlp, mlp.n_iter_))

## 四、numpy 手写前馈网络（理解反向传播）

In [ ]:
Xs = (X - X.mean(0)) / (X.std(0)+1e-9)
net = deep.NumpyNet(Xs.shape[1], 16, lr=0.05)
net.fit(Xs[:cut], y[:cut], epochs=100)
p_net = net.predict(Xs[cut:])
ic_net = pd.Series(p_net).corr(pd.Series(y[cut:]), method='spearman')
print('NumpyNet 测试 IC=%.4f 末轮loss=%.4f' % (ic_net, net.loss_hist[-1]))

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False
fig, ax = plt.subplots(figsize=(8.5,4.4))
ax.plot(net.loss_hist, color='#4f46e5', label='NumpyNet 训练MSE')
ax.plot(pd.Series(mlp.loss_curve_).rolling(5).mean(), color='#f87171',
        label='MLP loss(平滑)')
ax.set_xlabel('迭代'); ax.set_ylabel('MSE'); ax.set_title('训练损失曲线'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6.5,4.2))
ax.bar(['MLP','NumpyNet'], [ic_mlp, ic_net], color=['#60a5fa','#fbbf24'])
ax.axhline(0, color='#333', lw=0.8); ax.set_ylabel('测试IC'); ax.set_title('神经网络测试 IC 对比')
plt.tight_layout(); plt.show()

## 五、小结与面试要点

**面试高频：**
- LSTM 为什么能处理长序列？（答：门控+细胞状态缓解梯度消失，选择性记忆遗忘）
- 自注意力公式与作用？（答：QK点积缩放softmax后加权V，建模全局依赖、可并行）
- GNN 怎么用在股票上？（答：以股票为节点、关系为边，消息传递聚合关联信息）
- 量化里为什么树模型常胜过深度模型？（答：金融信噪比低、样本有限，深度模型易过拟合，树模型归纳偏置更稳）
- 深度模型怎么防过拟合？（答：早停、正则、dropout、减少参数、严格外推验证）

## 动手练习
1. 用 sklearn MLP 训练并评估 IC；
2. 用 numpy 前馈网络跑通并观察损失下降；
3. 口述 LSTM/Transformer/GNN 的结构与差异。

---
下一节 M3.6 强化学习与生成式。